In [0]:
from pyspark.sql.types import StringType
from pyspark.sql import Window

from pyspark.sql.functions import trim, col, row_number, when, current_date, year



In [0]:
%python
df = spark.read.table('workspace.bronze.crm_prd_info')

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
df.display()

In [0]:
df = (
    df.withColumn(
        'prd_end_dt',
        when(
            (col('prd_end_dt').isNull()) &
            (year(col('prd_start_dt')) == 2013),
            current_date()
        ).otherwise(col('prd_end_dt'))
    )
)

df.display()

In [0]:
df = (
    df.withColumn(
        'prd_line',
        when(
            (col('prd_line') == 'S'),
            'Sport'
        ).
        when(
           (col('prd_line') == 'R'),
            'Road' 
        ).
        when(
           (col('prd_line') == 'M'),
            'Mountain' 
        ).
        when(
           (col('prd_line') == 'T'),
            'Touring' 
        ).otherwise('Unknown')
    )
)

df.display()

In [0]:
RENAME_MAP = {
    "prd_id":"product_id",
    "prd_key":"product_key",
    "prd_nm":"product_name",
    "prd_cost":"product_cost",
    "prd_line":"product_line",
    "prd_start_dt":"product_start_date",
    "prd_end_dt":"product_end_date",
}

In [0]:

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

df.display()

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("silver.crm_products")